# Stress Testing

Load a trained model checkpoint and scaler from `testing.ipynb`, then run stress, regime, and tail-risk evaluation.

In [22]:
import importlib
import pickle
from pathlib import Path

import pandas as pd
import torch

from models.hybrid_v1 import HybridNetV1
from models.hybrid_v2 import HybridNetV2
from models.hybrid_v3 import HybridNetV3
import utils.evaluation

importlib.reload(utils.evaluation)

representative_sample = utils.evaluation.representative_sample
stress_scenarios = utils.evaluation.stress_scenarios
regime_test = utils.evaluation.regime_test
tail_risk_analysis = utils.evaluation.tail_risk_analysis

In [23]:
MODEL_ARTIFACT = Path("artifacts/trained_model_checkpoint.pt")
SCALER_ARTIFACT = Path("artifacts/trained_scaler.pkl")

checkpoint = torch.load(MODEL_ARTIFACT, map_location="cpu")

with open(SCALER_ARTIFACT, "rb") as f:
    scaler = pickle.load(f)

{k: checkpoint[k] for k in checkpoint if k != "state_dict"}

{'model_name': 'HybridNetV3',
 'state_dict': OrderedDict([('shared.0.weight',
               tensor([[-0.0877,  0.0146, -0.0692,  ..., -0.0925,  0.0163, -0.2690],
                       [-0.4723,  0.1039, -0.0333,  ..., -0.2128, -0.0020,  0.0418],
                       [-0.0728,  0.0047, -0.1618,  ..., -0.0832, -0.2225, -0.4614],
                       ...,
                       [ 0.1887,  0.1272, -0.1384,  ...,  0.0214, -0.2676,  0.0905],
                       [ 0.0012, -0.1354,  0.2065,  ..., -0.2327, -0.0959,  0.3173],
                       [ 0.1563, -0.1769,  0.1189,  ...,  0.0331,  0.1107, -0.6270]])),
              ('shared.0.bias',
               tensor([-1.6389e-01, -2.2106e-01,  1.0119e-01,  6.0168e-03, -2.5562e-01,
                        2.0486e-01,  1.2092e-01, -2.2484e-01,  1.3194e-01, -1.1927e-01,
                       -2.6486e-01,  5.4139e-02,  4.7556e-02, -2.3515e-02,  1.5130e-01,
                       -1.1776e-01,  2.2438e-01, -2.1711e-01, -1.4241e-01, -4.7076e-0

In [24]:
MODEL_REGISTRY = {
    "HybridNetV1": HybridNetV1,
    "HybridNetV2": HybridNetV2,
    "HybridNetV3": HybridNetV3,
}

model_name = checkpoint["model_name"]
features = checkpoint["features"]
input_dim = checkpoint["input_dim"]
sigma_index = checkpoint.get("sigma_index", features.index("sigma"))

if model_name == "HybridNetV3":
    model = MODEL_REGISTRY[model_name](input_dim=input_dim, sigma_index=sigma_index)
else:
    model = MODEL_REGISTRY[model_name](input_dim=input_dim)

model.load_state_dict(checkpoint["state_dict"])
model.eval()

print(f"Loaded model: {model_name}")
print(f"Split date used during training: {checkpoint['split_date']}")
print(f"Training fraction used: {checkpoint['train_size']:.0%}")

Loaded model: HybridNetV3
Split date used during training: 2022-10-07
Training fraction used: 70%


In [25]:
DATA_FILE = checkpoint.get("data_file", "data/processed/AsianPaints_Model_Data_2019_24.csv")
df = pd.read_csv(DATA_FILE)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.replace([float("inf"), float("-inf")], pd.NA).dropna().sort_values("Date").reset_index(drop=True)

print(f"Loaded data from: {DATA_FILE}")
print(f"Rows available for stress testing: {len(df):,}")
df.head()

Loaded data from: data/processed/AsianPaints_Model_Data_2019_24.csv
Rows available for stress testing: 212,343


,Date,S,K,T,r,sigma,Market_Price,vix,vol_ma,bs_price,...,log_moneyness,sqrt_T,sigma_sqrt_T,delta,gamma,vega,theta,vix_ratio,time_vol,vega_scaled
0,2019-01-29,1315.547363,980.0,0.082192,0.067,0.148314,418.95,18.43,1706227.8,339.848416,...,0.294456,0.286691,0.042520,1.000000,9.577295e-14,2.020537e-09,-65.299414,0.008047,0.042520,4.822860e-12
1,2019-01-29,1315.547363,1040.0,0.158904,0.067,0.148314,365.80,18.43,1706227.8,284.472545,...,0.235032,0.398628,0.059122,0.999986,8.070134e-07,3.291633e-02,-68.956170,0.008047,0.059122,8.998451e-05
2,2019-01-29,1315.547363,1020.0,0.082192,0.067,0.148314,379.20,18.43,1706227.8,300.068084,...,0.254450,0.286691,0.042520,1.000000,4.789070e-11,1.010358e-06,-67.964697,0.008047,0.042520,2.664446e-09
3,2019-01-29,1315.547363,1040.0,0.082192,0.067,0.148314,359.30,18.43,1706227.8,280.177919,...,0.235032,0.286691,0.042520,1.000000,7.107685e-10,1.499520e-05,-69.297350,0.008047,0.042520,4.173447e-08
4,2019-01-29,1315.547363,1060.0,0.082192,0.067,0.148314,339.40,18.43,1706227.8,260.287754,...,0.215984,0.286691,0.042520,1.000000,8.182527e-09,1.726281e-04,-70.630127,0.008047,0.042520,5.086272e-07


In [26]:
# Speed / fidelity controls
USE_SAMPLING = True
STRESS_SAMPLE_SIZE = 25000
REGIME_SAMPLE_SIZE = 25000
TAIL_SAMPLE_SIZE = 50000
RANDOM_STATE = 42

print(f"Sampling enabled: {USE_SAMPLING}")
print(f"Stress sample size: {STRESS_SAMPLE_SIZE}")
print(f"Regime sample size: {REGIME_SAMPLE_SIZE}")
print(f"Tail sample size: {TAIL_SAMPLE_SIZE}")

Sampling enabled: True
Stress sample size: 25000
Regime sample size: 25000
Tail sample size: 50000


In [27]:
scenario_results = stress_scenarios(
    model,
    df,
    features,
    scaler,
    max_rows=STRESS_SAMPLE_SIZE if USE_SAMPLING else None,
    random_state=RANDOM_STATE,
)

regime_results = regime_test(
    model,
    df,
    features,
    scaler,
    max_rows=REGIME_SAMPLE_SIZE if USE_SAMPLING else None,
    random_state=RANDOM_STATE,
)

tail_results = tail_risk_analysis(
    model,
    df,
    features,
    scaler,
    max_rows=TAIL_SAMPLE_SIZE if USE_SAMPLING else None,
    random_state=RANDOM_STATE,
)

In [28]:
pd.DataFrame(scenario_results).T

,n_rows,hybrid_mean,hybrid_std,bs_mean,bs_std
base,25000.0,205.882462,194.366013,166.664205,168.991757
low_vol,25000.0,200.361176,199.494568,142.019501,176.062594
high_vol,25000.0,220.615494,201.314133,240.601302,162.441784
deep_ITM,25000.0,863.534729,356.799286,798.110118,280.744895
deep_OTM,25000.0,138.594727,184.721741,10.192886,26.098988
near_expiry,25000.0,228.334595,266.751312,127.386579,172.433101
market_crash,25000.0,214.918945,189.625336,24.136603,41.369699


In [29]:
pd.DataFrame(regime_results).T

,n_rows,hybrid_rmse,bs_rmse
low_vix,25000.0,18.725117,164.122361
high_vix,25000.0,12.178687,190.096961


In [30]:
pd.Series(tail_results)

hybrid_95     0.169771
hybrid_99     0.413261
bs_95        22.371529
bs_99        61.461949
dtype: float64